Author: Tatum

In [2]:
# Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import CategoricalDtype
from pathlib import Path
from functools import reduce

### User input required
Put the data path on your system in the cell below

In [4]:
# Enter data path

data_path = r"/Users/tatumthomas/Northwestern University/Arvind Krishna - All Calls by Month"


### User input ends

### Reading all filenames in the data folder

In [9]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))

### Reading first 5 rows of all data files
The code chunk below reads the first 5 rows of all data files. This is to check the columns that are present in all the data files.

In [12]:
i=0; df = []
for f in files:
    if f.suffix.lower() == ".csv":
        df.append(pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False, nrows = 5))
    else:  # .xlsx
        df.append(pd.read_excel(f, sheet_name=0, header=0, dtype=str, nrows = 5))
    #df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df[i].shape)
    i = i + 1

0 April 2024 (5, 63)
1 April 2025 (5, 63)
2 August 2024 (5, 57)
3 August 2025 (5, 63)
4 December 2024 (5, 55)
5 February 2025 (5, 63)
6 January 2025 (5, 64)
7 July 2024 (5, 55)
8 July 2025 (5, 63)
9 June 2024 (5, 63)
10 June 2025 (5, 63)
11 March 2025 (5, 66)
12 May 2024 (5, 63)
13 May 2025 (5, 63)
14 November 2024 (5, 63)
15 October 2024 (5, 55)
16 September 2024 (5, 55)
17 September 2025 (5, 69)


The code chunk below identifies the columns missing in at least one DataFrame.

In [14]:
all_cols = reduce(lambda x, y: x | set(y.columns), df[1:], set(df[0].columns))
common_cols = reduce(lambda x, y: x & set(y.columns), df[1:], set(df[0].columns))
common_cols
not_in_all = all_cols - common_cols
print("Columns missing from at least one dataframe:", not_in_all)

Columns missing from at least one dataframe: {'Queue Type', 'External caller ID number', 'Public Called IP Address', 'Column1', 'Hold Duration', 'Public Calling IP Address', 'Call Recording Result', 'Answered Elsewhere', 'PSTN vendor name2', 'Recall Type', 'Original called party UUID', 'Call Recording Platform Name', 'Device owner UUID', 'User', 'Redirecting party UUID', 'Original reason2', 'Call Recording Trigger', 'Auto Attendant Key Pressed'}


The code chunk below prints the columns present in all the data files.

In [16]:
print(common_cols)

{'Location', 'Direction', 'Answer time', 'Client version', 'Department ID', 'Final remote sessionID', 'PSTN vendor Org ID', 'Inbound trunk', 'Related reason', 'Site UUID', 'Redirect reason', 'Site main number', 'Remote SessionID', 'Call transfer time', 'Org UUID', 'Call outcome reason', 'Route group', 'Ring duration', 'Authorization code', 'Local call ID', 'Model', 'OS type', 'Network call ID', 'Called number', 'International Country', 'Sub client type', 'Transfer related call ID', 'Release time', 'User type', 'Outbound trunk', 'Client type', 'Redirecting number', 'Start time', 'Correlation ID', 'Call ID', 'Report time', 'Local SessionID', 'Remote call ID', 'Call type', 'PSTN legal entity', 'Final local sessionID', 'Related call ID', 'Releasing party', 'Call outcome', 'Device Mac', 'PSTN vendor name', 'Original reason', 'Report ID', 'Answered', 'User UUID', 'Duration', 'PSTN provider ID', 'User number', 'Site timezone', 'Answer Indicator'}


### Reading all the data files
All the datafiles are read with the common columns read first.

In [18]:
df_main = pd.DataFrame(columns=list(common_cols))

In [19]:
i=0; 
for f in files:
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=0, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)
    i = i + 1

0 April 2024 (56662, 63)
1 April 2025 (63636, 63)
2 August 2024 (63262, 57)
3 August 2025 (57071, 63)
4 December 2024 (49445, 55)
5 February 2025 (63669, 63)
6 January 2025 (62623, 64)
7 July 2024 (62292, 55)
8 July 2025 (60438, 63)
9 June 2024 (56763, 63)
10 June 2025 (54598, 63)
11 March 2025 (59149, 66)
12 May 2024 (62944, 63)
13 May 2025 (55428, 63)
14 November 2024 (49953, 63)
15 October 2024 (62354, 55)
16 September 2024 (61250, 55)
17 September 2025 (56571, 69)


### Converting date to datetime format

In [21]:
df_main["Start time"] = pd.to_datetime(df_main["Start time"], utc=True)

In [22]:
df_main.to_csv("combined_calls.csv", index=False)

In [23]:

df_main["Start time"] = df_main["Start time"].dt.tz_convert("America/Chicago").dt.tz_localize(None)

In [24]:
df_main["Start time"].head()

0   2024-04-30 18:58:53.988
1   2024-04-30 18:56:37.386
2   2024-04-30 18:54:59.099
3   2024-04-30 18:54:59.099
4   2024-04-30 18:54:52.336
Name: Start time, dtype: datetime64[ns]

In [25]:
# Extract hour of day
df_main["Start Time Hour"] = df_main["Start time"].dt.hour
print(df_main[["Start time", "Start Time Hour"]].head())


               Start time  Start Time Hour
0 2024-04-30 18:58:53.988               18
1 2024-04-30 18:56:37.386               18
2 2024-04-30 18:54:59.099               18
3 2024-04-30 18:54:59.099               18
4 2024-04-30 18:54:52.336               18


In [39]:
df_main.to_csv("combined_calls_with_start_time_hour.csv", index=False)

### Front desk analysis 

In [70]:
FD   = "13122296300"   # front desk
MAIN = "13123411070"   # main line

df = df_main.copy()
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'[^0-9a-z]+', '_', regex=True)
)

# Parse times
for col in ["start_time", "answer_time", "release_time", "report_time"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

# Best-effort end_time
df["end_time"] = df["release_time"]
df["end_time"] = df["end_time"].fillna(df.get("report_time"))
df["end_time"] = df["end_time"].fillna(df.get("answer_time"))
df["end_time"] = df["end_time"].fillna(df.get("start_time"))

# Order legs inside a journey
df = df.sort_values(["correlation_id", "end_time"])
df["leg_idx"]     = df.groupby("correlation_id").cumcount()
df["last_leg_ix"] = df.groupby("correlation_id")["leg_idx"].transform("max")
df["is_last_leg"] = df["leg_idx"].eq(df["last_leg_ix"])


In [71]:
fd_inbound = df[(df["called_number"] == FD) & (df["direction"] == "TERMINATING")]

# Distinct journeys that ever hit FD inbound
fd_inbound_journeys = fd_inbound["correlation_id"].dropna().unique()
n_fd_inbound_journeys = len(fd_inbound_journeys)
print(f"Unique journeys with an FD inbound leg: {n_fd_inbound_journeys:,}")

# Among inbound journeys, check if the last leg is an FD leg
fd_inbound_last = (
    df[df["correlation_id"].isin(fd_inbound_journeys)]
      .groupby("correlation_id", as_index=False)
      .apply(lambda g: ((g["is_last_leg"]) & (g["called_number"] == FD)).any())
)

# Clean up the result
fd_inbound_last = fd_inbound_last.rename(columns={None: "fd_is_last"})
fd_inbound_last_count = fd_inbound_last["fd_is_last"].sum()
pct_fd_inbound_last = (fd_inbound_last_count / n_fd_inbound_journeys * 100) if n_fd_inbound_journeys else 0

print(f"FD is the final leg in inbound journeys: {fd_inbound_last_count:,}/{n_fd_inbound_journeys:,} "
      f"({pct_fd_inbound_last:.2f}%).")



Unique journeys with an FD inbound leg: 18,591
FD is the final leg in inbound journeys: 11,891/18,591 (63.96%).


/var/folders/0k/z0xdft5548x02pfzbdd0x6w40000gn/T/ipykernel_85118/2431917630.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: ((g["is_last_leg"]) & (g["called_number"] == FD)).any())


In [72]:
# Journey-level attribution of “how the FD leg arrived”
journey_first_fd_leg = (
    fd_inbound
    .sort_values(["correlation_id", "end_time"])
    .groupby("correlation_id", as_index=False)
    .first()  # first FD-inbound leg within that journey
)

from_main_journeys = (journey_first_fd_leg["redirecting_number"] == MAIN).sum()
from_blank_journeys = journey_first_fd_leg["redirecting_number"].isna().sum()
from_other_journeys = len(journey_first_fd_leg) - from_main_journeys - from_blank_journeys

print("Inbound journeys → first FD leg source (journey-level):")
print(f"  From MAIN (redirecting_number == MAIN): {from_main_journeys:,}")
print(f"  Direct/Unknown (redirecting_number is blank): {from_blank_journeys:,}")
print(f"  Other redirecting numbers: {from_other_journeys:,}")


Inbound journeys → first FD leg source (journey-level):
  From MAIN (redirecting_number == MAIN): 15,794
  Direct/Unknown (redirecting_number is blank): 2,485
  Other redirecting numbers: 312


In [73]:
def next_hop_after_last_fd(g):
    fd_rows = g[(g["called_number"] == FD) & (g["direction"] == "TERMINATING")]
    if fd_rows.empty:
        return None
    last_fd_idx = fd_rows["leg_idx"].max()
    candidate = g[g["leg_idx"] == last_fd_idx + 1]
    if candidate.empty:
        return None
    r = candidate.iloc[0]
    return f'{r.get("direction","")}|to:{r.get("called_number","")}|from:{r.get("user_number","")}'

nonfinal_inbound = (
    df[df["correlation_id"].isin(fd_inbound_journeys)]
      .groupby("correlation_id", group_keys=False)
      .apply(lambda g: (g["is_last_leg"] & (g["called_number"] == FD)).any())
)

nonfinal_ids = set(fd_inbound_journeys) - set(nonfinal_inbound[nonfinal_inbound].index)
nhops = (
    df[df["correlation_id"].isin(nonfinal_ids)]
      .groupby("correlation_id", group_keys=False)
      .apply(next_hop_after_last_fd)
      .dropna()
)

print("\nTop immediate next hops after FD (in inbound journeys where FD is not last):")
print(nhops.value_counts().head(15))


/var/folders/0k/z0xdft5548x02pfzbdd0x6w40000gn/T/ipykernel_85118/4174689340.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g["is_last_leg"] & (g["called_number"] == FD)).any())



Top immediate next hops after FD (in inbound journeys where FD is not last):
TERMINATING|to:13123411070|from:13123411070    3532
ORIGINATING|to:13122296300|from:13123411070    1070
TERMINATING|to:13125068646|from:13125068646     404
ORIGINATING|to:13123478300|from:13124235938     393
TERMINATING|to:13123478300|from:13123478300     298
TERMINATING|to:1180|from:1180                   287
TERMINATING|to:1182|from:1182                   140
TERMINATING|to:13125068647|from:13125068647     107
TERMINATING|to:13122296346|from:13122296346      27
TERMINATING|to:13123478342|from:13123478342      27
ORIGINATING|to:13123478300|from:13122296073      19
TERMINATING|to:13123478375|from:13123478375      14
ORIGINATING|to:13125068649|from:13124235938      13
ORIGINATING|to:13123478300|from:13122296346      12
ORIGINATING|to:2302|from:13123411070             12
Name: count, dtype: int64


/var/folders/0k/z0xdft5548x02pfzbdd0x6w40000gn/T/ipykernel_85118/4174689340.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(next_hop_after_last_fd)


In [74]:

# Total unique journeys that reached the front desk (inbound)
total_fd_inbound_journeys = fd_inbound["correlation_id"].nunique()

# Unique journeys where redirecting_number is blank/NaN (direct dial)
direct_fd_journeys = fd_inbound[fd_inbound["redirecting_number"].isna()]["correlation_id"].nunique()

# Compute percentage
pct_direct_fd = (direct_fd_journeys / total_fd_inbound_journeys * 100) if total_fd_inbound_journeys else 0

print(f"Direct-to-FD calls: {direct_fd_journeys:,} of {total_fd_inbound_journeys:,} "
      f"({pct_direct_fd:.2f}% of FD inbound journeys)")


Direct-to-FD calls: 2,488 of 18,591 (13.38% of FD inbound journeys)


## OLD ANALYSIS (DO NOT USE)

In [70]:
FD   = "13122296300"   # front desk number
MAIN = "13123411070"   # main number

df = df_main.copy()
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'[^0-9a-z]+', '_', regex=True)
)

# PARSE TIMES 
for col in ["start_time", "answer_time", "release_time", "report_time"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

# USING END TIMES
df["end_time"] = df["release_time"]
df["end_time"] = df["end_time"].fillna(df.get("report_time"))
df["end_time"] = df["end_time"].fillna(df.get("answer_time"))
df["end_time"] = df["end_time"].fillna(df.get("start_time"))

# ORDER LEGS WITHIN A CALL JOURNEY 
df = df.sort_values(["correlation_id", "end_time"])
df["leg_idx"]    = df.groupby("correlation_id").cumcount()
df["last_leg_ix"] = df.groupby("correlation_id")["leg_idx"].transform("max")
df["is_last_leg"] = df["leg_idx"].eq(df["last_leg_ix"])

# FRONT DESK SUBSETS
fd_inbound  = df[(df["called_number"] == FD) & (df["direction"] == "TERMINATING")]
fd_outbound = df[(df.get("user_number") == FD) & (df["direction"] == "ORIGINATING")]

# FRONT DESK AS DESTINATION (INBOUND)
print(f"Front desk appears as a call *destination* in {len(fd_inbound):,} legs.")
print("\nTop redirecting numbers → front desk:")
print(fd_inbound["redirecting_number"].value_counts(dropna=False).head(10))

#FRONT DESK AS CALLER (OUTBOUND)
print(f"\nFront desk appears as *caller* in {len(fd_outbound):,} legs.")
print("Top numbers the front desk calls:")
print(fd_outbound["called_number"].value_counts(dropna=False).head(10))

#IS THE FRONT DESK THE FINAL LEG (USING END_TIME)?
fd_total_journeys = df.loc[df["called_number"] == FD, "correlation_id"].nunique()
fd_last_journeys  = df.loc[(df["called_number"] == FD) & (df["is_last_leg"]), "correlation_id"].nunique()
fd_last_pct       = (fd_last_journeys / fd_total_journeys * 100) if fd_total_journeys else 0

print(f"\nFront desk is the final leg in {fd_last_journeys:,} of {fd_total_journeys:,} call journeys "
      f"({fd_last_pct:.2f}%).")

# HOW OFTEN DOES MAIN → FRONT DESK OCCUR?
# Count FD inbound legs whose *last redirecting number* is MAIN
fd_inbound_from_main = fd_inbound.loc[fd_inbound["redirecting_number"] == MAIN]
legs_main_to_fd = len(fd_inbound_from_main)

# Count unique journeys where an FD inbound leg has redirecting_number == MAIN
journeys_main_to_fd = fd_inbound_from_main["correlation_id"].nunique()
print(f"\nLegs with MAIN → FD (by redirecting_number): {legs_main_to_fd:,}")
print(f"Journeys containing MAIN → FD: {journeys_main_to_fd:,}")

# WITHIN A JOURNEY, DOES FD EVER CALL BACK OUT TO MAIN? (FD → MAIN)
journeys_fd_to_main = df.groupby("correlation_id").apply(
    lambda g: ((g.get("user_number") == FD) & (g["called_number"] == MAIN)).any()
).sum()

print(f"Journeys containing FD → MAIN (FD placing a call to MAIN): {journeys_fd_to_main:,}")

# COMMON NEXT HOPS AFTER FD WHEN FD IS NOT LAST LEG
# For journeys where FD is NOT the last leg, find the leg right after the FD leg (by end_time order)
def next_hop_after_fd(group):
    # all FD legs in this journey
    fd_rows = group[group["called_number"] == FD]
    if fd_rows.empty:
        return None
    # take the last FD leg in this journey (by end_time order)
    last_fd_ix = fd_rows["leg_idx"].max()
    # the immediate next leg after the last FD leg (if any)
    candidate = group[group["leg_idx"] == last_fd_ix + 1]
    if candidate.empty:
        return None
    row = candidate.iloc[0]
    # return where it went next
    return f'{row.get("direction", "")}|to:{row.get("called_number", "")}|from:{row.get("user_number", "")}'

non_last_fd_journeys = df.loc[(df["called_number"] == FD) & (~df["is_last_leg"]), "correlation_id"].unique()
next_hops = (
    df[df["correlation_id"].isin(non_last_fd_journeys)]
      .groupby("correlation_id", group_keys=False)
      .apply(next_hop_after_fd)
)

next_hops = next_hops.dropna()
print("\nTop immediate next hops after the last FD leg (for journeys where FD was not final):")
print(pd.Series(next_hops).value_counts().head(15))


Front desk appears as a call *destination* in 19,344 legs.

Top redirecting numbers → front desk:
redirecting_number
13123411070    16099
NaN             2515
13125068647      355
13125068646      173
13123478342      105
13127536357       41
13124312299       20
13125068649       16
13127536356       14
13122296080        4
Name: count, dtype: int64

Front desk appears as *caller* in 13,718 legs.
Top numbers the front desk calls:
called_number
1180           5836
1182           2132
13123478300    1611
13122296346     314
13125068649     218
13123478375     157
13123478326     157
13123478312     131
13127221813     105
13122296341      88
Name: count, dtype: int64

Front desk is the final leg in 11,894 of 18,595 call journeys (63.96%).

Legs with MAIN → FD (by redirecting_number): 16,099
Journeys containing MAIN → FD: 15,903


/var/folders/0k/z0xdft5548x02pfzbdd0x6w40000gn/T/ipykernel_64612/3152477745.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  journeys_fd_to_main = df.groupby("correlation_id").apply(


Journeys containing FD → MAIN (FD placing a call to MAIN): 7

Top immediate next hops after the last FD leg (for journeys where FD was not final):
TERMINATING|to:13123411070|from:13123411070    4497
TERMINATING|to:13125068646|from:13125068646     404
ORIGINATING|to:13123478300|from:13124235938     399
TERMINATING|to:13123478300|from:13123478300     377
TERMINATING|to:1180|from:1180                   287
TERMINATING|to:1182|from:1182                   140
TERMINATING|to:13125068647|from:13125068647     108
TERMINATING|to:13123478342|from:13123478342      31
TERMINATING|to:13122296346|from:13122296346      27
ORIGINATING|to:13123478300|from:13122296073      19
ORIGINATING|to:14154490512|from:13125068646      15
TERMINATING|to:13123478375|from:13123478375      14
ORIGINATING|to:13125068649|from:13124235938      13
ORIGINATING|to:13123478300|from:13122296346      12
ORIGINATING|to:2302|from:13123411070             12
Name: count, dtype: int64


/var/folders/0k/z0xdft5548x02pfzbdd0x6w40000gn/T/ipykernel_64612/3152477745.py:89: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(next_hop_after_fd)
